# Method 3 (short version) — Huber + Laplacian, solved by IRLS

Same code and results as `method_3.ipynb`, without the vocabulary, the detailed IRLS derivation and the loss
illustration; see the full version for those.

Notebook map:
1. Load the data, the group's shared code, evaluation helpers.
2. Assumptions, loss, IRLS, parameter choice (summary); code and the functions for the group.
3. Final result on the real data, `results/method_3/xhat_3.npy`, figures in `figures/method_3/`.
4. Result row for the comparison table.
5. Write-up (assignment Section VI).

Appendices: A. `cv_score`; B. synthetic data; C. comparison with Tikhonov on 100 graphs; D. IRLS checks.

## 1. Data and shared code

In [ ]:
# ---- Assignment Sections III/IV: load G, y, positions ----------------------
# NOTE: replace the representative student ID with the group's REAL ID (use the same
# ID as the group notebook), otherwise you get a different graph.
import hashlib, json
from pathlib import Path
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# Repository root = the folder that holds public_cases.json (the notebook runs from any subfolder).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "public_cases.json").exists())
FIG_DIR = ROOT / "figures" / "method_3"        # figures of this method
RES_DIR = ROOT / "results" / "method_3"        # xhat_3.npy
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

MSSV_DAI_DIEN = "20260001"          # <-- change to the group's real representative student ID

def assign_mssv_to_case(student_id, cases):
    # Same deterministic ID -> case mapping as in the assignment notebook.
    sid = str(student_id).strip()
    if sid in cases:
        return sid
    case_ids = sorted(cases, key=lambda x: (0, int(x)) if x.isdecimal() else (1, x))
    k = len(case_ids)
    if sid.isdecimal():
        index = int(sid) % k
    else:
        index = int.from_bytes(hashlib.sha256(sid.encode("utf-8")).digest(), "big") % k
    return case_ids[index]

def get_problem(student_id, file_path=ROOT / "public_cases.json"):
    db = json.loads(Path(file_path).read_text(encoding="utf-8"))
    case = db["cases"][assign_mssv_to_case(student_id, db["cases"])]
    G = nx.Graph()
    for i, xy in enumerate(case["positions"]):
        G.add_node(i, pos=tuple(map(float, xy)))
    for u, v, w in case["edges"]:
        G.add_edge(int(u), int(v), weight=float(w))
    y = np.asarray(case["y"], dtype=float)
    assert nx.is_connected(G) and len(y) == G.number_of_nodes()
    return G, y

G, y = get_problem(MSSV_DAI_DIEN)
positions = nx.get_node_attributes(G, "pos")
n = len(y)
print("n nodes =", n, "| n edges =", G.number_of_edges())
print("std(y) = %.4f | range = %.2f .. %.2f" % (np.std(y), y.min(), y.max()))

In [ ]:
# ---- Shared "contract" block of the group ----------------------------------
# L = weighted graph Laplacian = D - W.  For any signal x:
#     x^T L x = sum_{(i,j)} w_ij (x_i - x_j)^2
# i.e. x^T L x is SMALL when nearby nodes have similar values (a "smooth" signal).
n = len(y)
L = nx.laplacian_matrix(G, nodelist=range(n), weight="weight").toarray()

# (a) Synthetic data: known x, used to measure RMSE.
def make_synthetic(G, y, seed):
    rng = np.random.default_rng(seed)
    lam, U = np.linalg.eigh(L)
    # x = random combination of the 5 lowest-frequency eigenvectors (a smooth signal)
    x = U[:, :5] @ (rng.normal(size=5) / (1 + lam[:5]))
    # rescale so that x has the same median / standard deviation as the real y
    x = np.median(y) + np.std(y) * x / np.std(x)
    sigma = 0.1 * np.std(y)
    y_syn = x + rng.normal(0, sigma, n)               # Gaussian noise
    idx = rng.choice(n, 3, replace=False)             # 3 outliers
    y_syn[idx] += rng.choice([-1, 1], 3) * rng.uniform(4, 8, 3) * sigma
    return x, y_syn

def rmse(a, b): return np.sqrt(np.mean((a - b) ** 2))

# (b) Choose parameters on the REAL data: hide 1/5 of the nodes and predict them again.
def cv_score(solve, y, params, k=5, seed=0):
    folds = np.random.default_rng(seed).permutation(n) % k
    err = []
    for f in range(k):
        mask = folds != f                              # True = observed
        x = solve(G, y, mask=mask, **params)
        err.append(np.abs(x[~mask] - y[~mask]))
    return np.median(np.concatenate(err))              # median => robust to outliers in y

In [ ]:
# ---- Evaluation helpers used in Section 4 and the appendices ----------------
def cv_mean(params, seeds=range(5)):
    # cv_score averaged over 5 splits — the group convention (lower = better).
    return np.mean([cv_score(solve_huber, y, params, seed=sd) for sd in seeds])

def cv_mean20(params, seeds=range(20)):
    # Same but 20 splits: use when COMPARING close parameter values, because with
    # 32 nodes a single split gives a very noisy cv_score (see Appendix A).
    return np.mean([cv_score(solve_huber, y, params, seed=sd) for sd in seeds])

def synth_rmse(params, seeds=range(20)):
    # RMSE against the TRUE x on synthetic data, one value per seed (0-19, group convention).
    out = []
    for sd in seeds:
        x_true, y_syn = make_synthetic(G, y, sd)
        out.append(rmse(solve_huber(G, y_syn, **params), x_true))
    return np.array(out)

## 2. Method (summary)

**Assumptions.** The true signal `x` is smooth on the graph; the observation is `y = x + ε + o` with small
(mostly Gaussian) noise `ε` at every node and a large outlier `o` at a few nodes. Edge weights are similarities
between sensors. Unlike method 1, the noise has heavy tails, so the data term must be robust.

**Objective.**

$$
\min_x \; \sum_i \rho_\delta(x_i-y_i) + \lambda\, x^\top L x,
\qquad
\rho_\delta(r)=
\begin{cases}
\tfrac12 r^2, & |r|\le\delta,\\
\delta|r|-\tfrac12\delta^2, & |r|>\delta.
\end{cases}
$$

- `ρ_δ` is quadratic for `|r| ≤ δ` (efficient under Gaussian noise) and linear for `|r| > δ` (the pull is
  capped at `δ`, so an outlier cannot drag the solution as the square does).
- `λxᵀLx = λΣ w_ij(x_i−x_j)²` smooths the signal; `λ > 0` guarantees a unique solution.
- `δ → ∞` gives Tikhonov (method 1); `δ → 0` approaches L1/LAD (method 4).

**IRLS.** Optimality condition `ψ_δ(r_i) + 2λ(Lx)_i = 0`; with `w_i = min(1, δ/|r_i|)` we have
`ψ_δ(r_i) = w_i r_i`, hence `(W + 2λL)x = Wy`, `W = diag(w_i)`. Iterate `x⁰ = y`; each iteration solves
`(W^k + 2λL)x^{k+1} = W^k y` and stops when `‖Δx‖/(1+‖x‖) < 10⁻¹⁰`. Each iteration is a weighted Tikhonov
problem (a symmetric positive definite system); the objective decreases monotonically, so the sequence
converges.

**Parameter choice.** `δ = α·s0` with `s0 = 1.4826·MAD(y_i − weighted mean of the neighbours) = 0.675`;
`α = 0.5` (i.e. `δ = 0.337`), `λ = 0.1`. Both parameters are selected on a grid by `cv_score` (mean over
20 splits) and checked with the synthetic RMSE — details in Appendix A.

### Code of the method

In [ ]:
# =============================================================================
# Method 3: Huber + Laplacian, solved by IRLS.
# The functions below are identical to the shared file src/method_3/method3_huber.py
# (Appendix D checks that both versions give the same result).
# =============================================================================
HUBER_DELTA_ALPHA = 0.5     # delta = 0.5 * s0 (value used in the group result)
DEFAULT_LAM = 0.1

def huber(r, delta):
    """Huber loss ρ_δ(r), elementwise.

    Input:  r (array), delta (float > 0 or np.inf).
    Output: array with the same shape as r.
    """
    a = np.abs(np.asarray(r, dtype=float))
    if np.isinf(delta):
        return 0.5 * a * a
    return np.where(a <= delta, 0.5 * a * a, delta * (a - 0.5 * delta))

def huber_weight(r, delta):
    """IRLS weight w = Huber'(r) / r (so that Huber'(r) = w·r).

    Input:  r (array), delta (float > 0 or np.inf).
    Output: array in (0, 1].
    """
    a = np.abs(np.asarray(r, dtype=float))
    if np.isinf(delta):
        return np.ones_like(a)
    return np.where(a <= delta, 1.0, delta / np.maximum(a, 1e-300))

def robust_scale(v):
    """Robust standard deviation: 1.4826 * MAD (median absolute deviation)."""
    v = np.asarray(v, dtype=float)
    return float(1.4826 * np.median(np.abs(v - np.median(v))))

def laplacian(G, n=None):
    """Weighted graph Laplacian L = D - W (dense)."""
    n = G.number_of_nodes() if n is None else n
    return nx.laplacian_matrix(G, nodelist=range(n), weight="weight").toarray()

def neighbour_mean(G, y, mask=None):
    """Weighted mean of the observed neighbours (NaN if no neighbour is observed)."""
    y = np.asarray(y, dtype=float)
    obs = np.ones(len(y), dtype=bool) if mask is None else np.asarray(mask, dtype=bool)
    m = np.full(len(y), np.nan)
    for i in range(len(y)):
        nb = [j for j in G.neighbors(i) if obs[j]]
        if nb:
            w = np.array([G[i][j]["weight"] for j in nb], dtype=float)
            m[i] = np.sum(w * y[nb]) / np.sum(w)
    return m

def noise_scale(G, y, mask=None):
    """Noise scale s0: 1.4826 * MAD of y_i minus the neighbours' mean.

    Only used to put delta on the data scale (delta = alpha * s0).
    """
    y = np.asarray(y, dtype=float)
    obs = np.ones(len(y), dtype=bool) if mask is None else np.asarray(mask, dtype=bool)
    r = y - neighbour_mean(G, y, mask=obs)
    valid = obs & ~np.isnan(r)
    s = robust_scale(r[valid])
    return max(float(s), 1e-8 * float(np.std(y[obs])) + 1e-12)

def huber_objective(G, y, x, delta, lam, mask=None, L=None):
    """Value of F(x) = sum_i Huber_delta(x_i - y_i) + lam * x^T L x."""
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    obs = np.ones(len(y), dtype=bool) if mask is None else np.asarray(mask, dtype=bool)
    L = laplacian(G, len(y)) if L is None else L
    return float(huber((x - y)[obs], delta).sum() + lam * x @ L @ x)

def solve_huber(G, y, mask=None, delta=None, lam=DEFAULT_LAM, x0=None,
                tol=1e-10, max_iter=1000, return_info=False):
    """Recover a graph signal with Huber loss + Laplacian regularisation, by IRLS.

    Input:
        G      networkx graph, nodes 0..n-1, every edge has a "weight" attribute.
        y      float array of length n: observed values.
        mask   bool array of length n, True = observed (default: all).
               False = hidden node: y_i is ignored and the value is reconstructed from the
               graph (used by cv_score).
        delta  Huber threshold (> 0, same units as y); np.inf = Tikhonov (method 1);
               None = 0.5 * noise_scale(G, y, mask) (group result).
        lam    weight of x^T L x (> 0); group result: 0.1.
        x0     starting point (default y); tol, max_iter: stopping rule.
        return_info  True = also return a dict (weights, iterations, objective history, KKT).

    Output:
        x_hat  array of length n (finite, exactly one value per node).
        info   dict if return_info=True: "weights", "n_iter", "objective", "kkt", "delta", "lam".

    Example:
        x_hat = solve_huber(G, y)                              # delta = 0.5*s0, lam = 0.1
        x_hat = solve_huber(G, y, delta=np.inf, lam=0.1)       # Tikhonov reference
        x_hat = solve_huber(G, y, mask=mask, delta=0.337, lam=0.1)  # inside cv_score
    """
    y = np.asarray(y, dtype=float)
    if y.ndim != 1 or not np.all(np.isfinite(y)):
        raise ValueError("y must be a finite 1-D array")
    n = len(y)
    obs = np.ones(n, dtype=bool) if mask is None else np.asarray(mask, dtype=bool)
    if obs.shape != y.shape:
        raise ValueError("mask must have the same length as y")
    if not obs.any():
        raise ValueError("at least one node must be observed")
    if lam <= 0:
        raise ValueError("lam must be > 0 (with lam = 0 the data term alone is minimised at x = y)")
    if delta is None:
        delta = HUBER_DELTA_ALPHA * noise_scale(G, y, mask=obs)
    delta = float(delta)
    if not (delta > 0):
        raise ValueError("delta must be > 0")

    L = laplacian(G, n)
    x = y.copy() if x0 is None else np.asarray(x0, dtype=float).copy()
    hist = []
    for it in range(1, max_iter + 1):
        w = obs * huber_weight(x - y, delta)               # step 2: weights from the current residuals
        x_new = np.linalg.solve(np.diag(w) + 2.0 * lam * L, w * y)   # step 3: one SPD system
        hist.append(huber_objective(G, y, x_new, delta, lam, mask=obs, L=L))
        step = np.linalg.norm(x_new - x) / (1.0 + np.linalg.norm(x_new))
        x = x_new
        if step < tol:
            break

    w = obs * huber_weight(x - y, delta)
    kkt = float(np.max(np.abs(w * (x - y) + 2.0 * lam * L @ x)))
    if not return_info:
        return x
    return x, {"x": x, "weights": w, "n_iter": it, "objective": hist, "kkt": kkt,
               "delta": delta, "lam": float(lam)}

def solve_tikhonov(G, y, mask=None, lam=DEFAULT_LAM):
    """Method 1 reference: min sum_i (x_i-y_i)^2 + lam * x^T L x.

    Closed form: (M + 2*lam*L) x = M y,  M = diag(mask).
    Identical to solve_huber(..., delta=np.inf).
    """
    y = np.asarray(y, dtype=float)
    n = len(y)
    obs = np.ones(n, dtype=bool) if mask is None else np.asarray(mask, dtype=bool)
    if not obs.any():
        raise ValueError("at least one node must be observed")
    if lam <= 0:
        raise ValueError("lam must be > 0")
    L = laplacian(G, n)
    M = obs.astype(float)
    return np.linalg.solve(np.diag(M) + 2.0 * lam * L, M * y)

### Functions for the group

| function | input | output |
|---|---|---|
| `solve_huber(G, y, mask=None, delta=None, lam=0.1, ..., return_info=False)` | graph, `y` (length n), optional `mask` (True = observed), `delta`, `lam` | `x_hat` (length n); with `return_info=True` also a dict of weights/iterations/KKT |
| `solve_tikhonov(G, y, mask=None, lam=0.1)` | as above | `x_hat` of method 1 (reference) |
| `noise_scale(G, y, mask=None)` | graph, `y` | `s0` — noise scale, used to set `delta = alpha * s0` |

Using it in another method: `x_hat = solve_huber(G, y, mask=mask, delta=0.337, lam=0.1)`, and call it again
inside every `cv_score` fold. The weights `info["weights"]` can be used as a per-node "suspicion score":
`w_i < 0.5` means the node is treated as anomalous.

In [ ]:
# ---- Example + self-checks --------------------------------------------------
# s0 = noise scale estimated from the data (Section 2.5)
S0 = noise_scale(G, y)
DELTA_EXAMPLE = 0.5 * S0
LAM_EXAMPLE = 0.1

x_demo, info = solve_huber(G, y, delta=DELTA_EXAMPLE, lam=LAM_EXAMPLE, return_info=True)
print("s0 = %.4f -> delta = %.4f | lam = %.2f" % (S0, info["delta"], info["lam"]))
print("IRLS iterations =", info["n_iter"], "| KKT max|g| = %.2e" % info["kkt"])
print("down-weighted nodes (w < 1):", int(np.sum(info["weights"] < 0.999)))
print("strongly down-weighted nodes (w < 0.5):", np.flatnonzero(info["weights"] < 0.5).tolist())

# 1) The objective never increases (majorize–minimize property of IRLS).
obj = np.array(info["objective"])
assert np.all(np.diff(obj) <= 1e-12), "the objective must decrease monotonically"
print("objective: %.6f -> %.6f, monotone decrease: OK" % (obj[0], obj[-1]))

# 2) The solution satisfies the optimality (KKT) condition.
print("KKT = %.2e (close to 0 means an accurate solution)" % info["kkt"])

# 3) With delta = +inf, Huber becomes the square => must match the closed-form Tikhonov solution.
x_inf = solve_huber(G, y, delta=np.inf, lam=LAM_EXAMPLE)
x_tik = solve_tikhonov(G, y, lam=LAM_EXAMPLE)
assert np.allclose(x_inf, x_tik, atol=1e-10), "delta = inf must match Tikhonov"
print("delta = inf matches Tikhonov: OK (max difference %.1e)" % np.max(np.abs(x_inf - x_tik)))

# 4) mask: hidden nodes do not use y but still get finite values.
mask_demo = np.random.default_rng(0).random(n) > 0.2
x_masked = solve_huber(G, y, mask=mask_demo, delta=DELTA_EXAMPLE, lam=LAM_EXAMPLE)
assert np.all(np.isfinite(x_masked))
print("mask: %d hidden nodes, solution still finite: OK" % int((~mask_demo).sum()))

## 3. Final result on the real data

Parameters: `δ = 0.5·s0 ≈ 0.337`, `λ = 0.1` (chosen in Appendix A). `S0` is computed in the example cell above.

In [ ]:
# ---- Final result on the REAL data -----------------------------------------
ALPHA = 0.5          # delta = ALPHA * s0  (chosen in Appendix A)
LAM = 0.1            # (chosen in Appendix A)
FINAL_PARAMS = {"delta": ALPHA * S0, "lam": LAM}

x_hat_real, info_real = solve_huber(G, y, return_info=True, **FINAL_PARAMS)
assert x_hat_real.shape == y.shape and np.all(np.isfinite(x_hat_real))

np.save(RES_DIR / "xhat_3.npy", x_hat_real)
print("FINAL_PARAMS =", {k: round(float(v), 6) for k, v in FINAL_PARAMS.items()})
print("s0 = %.4f | delta = %.4f | lam = %.2f" % (S0, FINAL_PARAMS["delta"], LAM))
print("IRLS iterations = %d | KKT = %.2e" % (info_real["n_iter"], info_real["kkt"]))
print("saved:", RES_DIR / "xhat_3.npy")

print("\nStrongly down-weighted nodes (w < 0.5):")
for i in np.flatnonzero(info_real["weights"] < 0.5):
    print("  node %2d: y = %7.3f -> x_hat = %7.3f | w = %.3f"
          % (i, y[i], x_hat_real[i], info_real["weights"][i]))
print("\nRMSE(x_hat, y) = %.4f  (for reference only, NOT the RMSE against the true x)"
      % rmse(x_hat_real, y))

In [ ]:
# ---- Figure: y and x_hat on the graph; strongly down-weighted nodes circled in red ----
vmin = min(y.min(), x_hat_real.min())
vmax = max(y.max(), x_hat_real.max())
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, vals, title in [(axes[0], y, "Observed y"),
                        (axes[1], x_hat_real, "Recovered x_hat (method 3)")]:
    nodes = nx.draw_networkx_nodes(G, positions, node_color=vals, cmap="viridis",
                                   vmin=vmin, vmax=vmax, node_size=380, ax=ax)
    nx.draw_networkx_edges(G, positions, alpha=0.3, ax=ax)
    nx.draw_networkx_labels(G, positions, font_size=8, ax=ax)
    ax.set_title(title)
    ax.axis("off")

suspect = np.flatnonzero(info_real["weights"] < 0.5)
nx.draw_networkx_nodes(G, positions, nodelist=list(suspect), node_color="none",
                       edgecolors="red", linewidths=2.5, node_size=520, ax=axes[1])
fig.colorbar(nodes, ax=axes, label="value", shrink=0.8)
fig.suptitle("Circled nodes: down-weighted by Huber (w < 0.5)", y=1.02)
plt.savefig(FIG_DIR / "fig_y_vs_xhat_method3.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---- Figure: IRLS convergence on the real data -----------------------------
obj = np.array(info_real["objective"])
gap = obj - obj[-1]
floor = 1e-13                         # below this level the difference is only floating-point round-off

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4))
axes[0].semilogy(np.arange(1, len(gap) + 1), np.maximum(gap, floor), "o-", ms=3)
axes[0].axhline(floor, color="gray", ls="--", lw=1, label="floating-point floor (1e-13)")
axes[0].set_ylim(floor / 2, 1)
axes[0].set_xlabel("iteration")
axes[0].set_ylabel("F(x^k) − F(x*)")
axes[0].set_title("Objective decreases monotonically (%d iterations)" % len(obj))
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Final weights vs residual magnitude: exactly the rule w = min(1, delta/|r|).
r_abs = np.abs(x_hat_real - y)
axes[1].scatter(r_abs, info_real["weights"], s=35, zorder=3)
rr = np.linspace(1e-3, r_abs.max() * 1.05, 400)
axes[1].plot(rr, np.minimum(1.0, FINAL_PARAMS["delta"] / rr), "k--", lw=1,
             label="w = min(1, δ/|r|), δ = %.3f" % FINAL_PARAMS["delta"])
axes[1].set_xlabel("|r_i| = |x_hat_i − y_i|")
axes[1].set_ylabel("weight w_i")
axes[1].set_title("Final weights: large deviations are down-weighted")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
plt.savefig(FIG_DIR / "fig_convergence_method3.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Result row for the group comparison table

One row per method, as agreed by the group: method name, best parameters, synthetic RMSE (mean ± standard
deviation over seeds 0–19), and `cv_score` on the real data. How the two numbers are computed and how to read
them: Appendix A (cv_score) and Appendix B (synthetic data).

In [ ]:
synth_final = synth_rmse(FINAL_PARAMS)
result_row = {
    "method": "3. Huber + Laplacian (IRLS)",
    "params": {"alpha": ALPHA, "delta": round(float(FINAL_PARAMS["delta"]), 4), "lam": LAM},
    "synthetic_RMSE_mean": round(float(synth_final.mean()), 4),
    "synthetic_RMSE_std": round(float(synth_final.std()), 4),
    "cv_score": round(float(cv_mean(FINAL_PARAMS)), 4),
}
print(result_row)

# Tikhonov (method 1) reference with the SAME lam, to isolate the contribution of the Huber part.
synth_l2 = synth_rmse({"delta": np.inf, "lam": LAM})
print("\nTikhonov reference with the same lam = %.2f:" % LAM)
print("  cv_score (5 splits) = %.4f | cv_score (20 splits) = %.4f"
      % (cv_mean({"delta": np.inf, "lam": LAM}), cv_mean20({"delta": np.inf, "lam": LAM})))
print("  synthetic RMSE = %.4f ± %.4f" % (synth_l2.mean(), synth_l2.std()))
print("\ncv_score of method 3 (5 splits, used in the group table) = %.4f"
      % cv_mean(FINAL_PARAMS))
print("Note: Appendix A selects parameters with the 20-split cv mean; the result row reports the 5-split convention.")

## 5. Write-up (assignment Section VI)

**1. Assumptions about the signal, the noise and nearby nodes.**
The true signal `x` is smooth on the graph: two nearby sensors (large `w_ij`) measure similar values, so
`xᵀLx = Σ w_ij(x_i−x_j)²` is small. The observation is `y = x + ε + o`, where `ε` is small noise at every
node (mostly Gaussian) and `o` is a large error at a few nodes (outliers), sparse and usually not adjacent.
Edge weights are similarities between sensors. Unlike method 1, the noise here has **heavy tails**: a single
badly read node can drag least squares off, so the data term must be robust. Consequence: `x` is the unknown
to recover, and `y` enters the problem only through `x−y`; the units of `y` only affect how `δ` is set.

**2. Objective.**
`min_x F(x) = Σ_i ρ_δ(x_i−y_i) + λxᵀLx`, with `ρ_δ(r) = ½r²` if `|r| ≤ δ` and `δ|r| − ½δ²` if `|r| > δ`.
The first term measures data fit: quadratic for small residuals (exploiting the Gaussian assumption), linear
for large ones, with derivative `ψ_δ(r) = min(|r|,δ)·sign(r)` capped at `δ`. Hence an outlier pulls the
solution with a force of at most `δ`, not proportional to its deviation as in L2. The second term `λxᵀLx`
measures roughness and smooths the signal; `λ > 0` also guarantees uniqueness. Quantitative example: with
`δ = 1`, a residual `r = 5` is penalised 12.5 by L2 (derivative 5), 5 by L1 (derivative 1) and 4.5 by Huber
(derivative 1). Two limiting cases: `δ → ∞` gives exactly the Tikhonov problem of method 1; `δ → 0` (with a
suitable λ) approaches L1/LAD of method 4 — method 3 is in between, and `δ` decides which side.

**3. Optimisation and parameter selection.**
The optimality condition is `ψ_δ(r_i) + 2λ(Lx)_i = 0` with `r_i = x_i − y_i`. Setting
`w_i = min(1, δ/|r_i|)` gives `ψ_δ(r_i) = w_i r_i`, so the condition becomes `W(x−y) + 2λLx = 0`, i.e.
`(W + 2λL)x = Wy`. `W` depends on the solution, so we iterate (IRLS): `x⁰ = y`; each iteration computes
`w_i` from `x^k` and solves `(W^k + 2λL)x^{k+1} = W^k y`. Each iteration is a weighted Tikhonov problem;
`W^k + 2λL` is symmetric positive definite (since `L` is positive semidefinite and at least one `w_i > 0`),
so it has a unique solution and one factorisation suffices. Each iteration is also a majorize–minimize step,
so `F(x^k)` never increases; `F` is convex with a unique minimiser, hence the sequence converges. On the real
data: 40 iterations, `F` decreases monotonically from 4.1572 to 3.9417, KKT norm `max|g| = 6.8×10⁻⁹`;
compared with the convex L-BFGS solver on the same objective, the objective differs by `8×10⁻¹²`
(Appendix D). Stopping rule: `‖Δx‖/(1+‖x‖) < 10⁻¹⁰`.
The parameters are chosen as follows: `δ = α·s0` with `s0 = 1.4826·MAD(y_i − neighbour mean) = 0.675`
(noise scale); grid `α ∈ {0.25, …, 2}`, `λ ∈ {0.025, …, 0.25}`, `cv_score` averaged over 20 splits; among the
points within 3% of the best `cv_score`, choose the one with the lowest synthetic RMSE. Result: `α = 0.5`,
`λ = 0.1`, i.e. `δ = 0.337`; `cv_score` = 0.3745 (20 splits) and 0.3309 (5 splits). We do not use the
textbook `δ = 1.345σ` because the real noise is large (`σ ≈ 0.6`, Appendix D) while the real outliers are only
moderate (about 2–4σ): a large threshold would almost ignore them. Conversely, `α = 0.25` is slightly better
on `cv_score` but much worse on synthetic RMSE (0.42 vs 0.26), so it is discarded.

**4. Experiments, visualisation and limitations.**
On the real data, the method down-weights 10 nodes, 5 of them strongly (`w < 0.5`: nodes 3, 12, 20, 28, 29);
node 28 has `y = 17.55` and is pulled to `x_hat = 18.80` (`w = 0.27`) — the same node that method 6 flags.
Compared with Tikhonov at the same `λ`, `cv_score` drops from 0.4043 to 0.3745 (20 splits; 0.3746 → 0.3309
with 5 splits), i.e. about 7–12%, so the robust part contributes in practice rather than being decorative.
On the 100 public graphs, with `λ` tuned per graph, Huber is only marginally better than Tikhonov (mean
`cv_score` 0.6046 vs 0.6080, winning 43/100) — the benefit depends on how many outliers a case has and is not
always large. Limitations: (a) `δ` and `λ` depend on the scale and noise level, so they must be re-tuned for a
different dataset; (b) Huber only down-weights outliers instead of removing them as method 6 does, so
extremely large outliers still have some influence; (c) too small a `δ` approaches L1 and loses Gaussian
efficiency, too large a `δ` returns to Tikhonov; (d) a large `λ` shrinks the signal amplitude (bias), and
`cv_score` only measures the hidden nodes, so it may pick a slightly larger `λ` than the RMSE target would;
(e) the smoothness assumption may fail at boundaries between two regions; (f) the group's synthetic generator
has several times less noise than the real data, so synthetic RMSE does not directly predict real RMSE — on
this generator Huber is about 0.024 RMSE worse than Tikhonov at the same `λ`, but when `δ` and `λ` are lowered
to match the synthetic noise, Huber is slightly better (0.233 vs 0.235); (g) each IRLS iteration solves a dense
`n×n` system — fine for 32 nodes, but large graphs need sparse factorisation.

---
# Appendices (optional reading)

- **Appendix A**: the `α` × `λ` grid on the real data and the selection rule.
- **Appendix B**: synthetic data with known `x` (seed 0 and seeds 0–19).
- **Appendix C**: comparison with Tikhonov on 100 public graphs.
- **Appendix D**: convergence/KKT checks, comparison with a convex solver, and a noise-level diagnostic.

## Appendix A. Parameter selection with `cv_score` (real data)

`cv_score` (shared code): 5 folds; hide 1/5 of the nodes, run the whole method on the remaining 4/5 (including
the weight computation), measure `|x_hat − y|` on the hidden nodes, and take the median over all of them.
`cv_score` is measured against noisy `y` (not the true `x`) and is very noisy on 32 nodes, so here we average
over **20 splits** when comparing close parameter values; the result row in Section 4 still reports the
group's **5-split** convention.

Grid: `α ∈ {0.25, 0.4, 0.5, 0.75, 1, 1.5, 2}` (i.e. `δ = α·s0`), `λ ∈ {0.025, 0.05, 0.075, 0.1, 0.15, 0.25}`.
For each point, the cell below computes `cv_score` (20 splits) and synthetic RMSE (20 seeds), then applies the
rule: among the points within 3% of the best `cv_score`, choose the lowest synthetic RMSE.

In [ ]:
ALPHAS = [0.25, 0.4, 0.5, 0.75, 1.0, 1.5, 2.0]
LAMS = [0.025, 0.05, 0.075, 0.1, 0.15, 0.25]
CV_SEEDS = range(20)

print("Computing the grid of %d points (%d splits x 5 folds for cv)..." % (len(ALPHAS) * len(LAMS), len(CV_SEEDS)))
grid = []
for a in ALPHAS:
    for l in LAMS:
        p = {"delta": a * S0, "lam": l}
        grid.append({"alpha": a, "lam": l,
                     "cv": float(cv_mean20(p, seeds=CV_SEEDS)),
                     "synth": float(synth_rmse(p).mean())})

print("\ncv_score (20-split mean) — rows alpha, columns lam")
print("alpha\\lam" + "".join("%9g" % l for l in LAMS))
for a in ALPHAS:
    print("%9.2f" % a + "".join("%9.4f" % [g for g in grid if g["alpha"] == a and g["lam"] == l][0]["cv"]
                                for l in LAMS))

best = min(grid, key=lambda g: g["cv"])
TOL_REL = 0.03
candidates = [g for g in grid if g["cv"] <= best["cv"] * (1 + TOL_REL)]
pick = min(candidates, key=lambda g: g["synth"])

print("\nbest cv_score on the grid: alpha = %.2f, lam = %.3f -> %.4f (synth %.4f)"
      % (best["alpha"], best["lam"], best["cv"], best["synth"]))
print("3%% threshold: cv <= %.4f -> %d candidate points" % (best["cv"] * (1 + TOL_REL), len(candidates)))
print("pick with lowest synthetic RMSE: alpha = %.2f, lam = %.3f -> cv %.4f, synth %.4f"
      % (pick["alpha"], pick["lam"], pick["cv"], pick["synth"]))
print("(point with the best cv: alpha = %.2f, lam = %.3f, synth %.4f)"
      % (best["alpha"], best["lam"], best["synth"]))

assert (pick["alpha"], pick["lam"]) == (ALPHA, LAM), "the rule must reproduce the parameters used in Section 3"
print("\nThe rule reproduces FINAL_PARAMS (alpha=%.2f, lam=%.2f): OK" % (ALPHA, LAM))

In [ ]:
# ---- Figure: parameter-selection curve -------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))

for l in LAMS:
    ys = [g["cv"] for g in grid if g["lam"] == l]
    axes[0].plot(ALPHAS, ys, "o-", ms=3.5, label="λ = %g" % l)
axes[0].axhline(best["cv"] * (1 + TOL_REL), color="gray", ls="--", lw=1, label="3% threshold")
axes[0].axvline(ALPHA, color="red", ls=":", label="chosen α = %.1f" % ALPHA)
axes[0].set_xlabel("α  (δ = α·s0)")
axes[0].set_ylabel("cv_score (20 splits)")
axes[0].set_title("Parameter-selection grid")
axes[0].legend(fontsize=8, ncol=2)
axes[0].grid(alpha=0.3)

M = np.array([[g["cv"] for g in grid if g["alpha"] == a] for a in ALPHAS])
im = axes[1].imshow(M, aspect="auto", origin="lower", cmap="viridis_r")
axes[1].set_xticks(range(len(LAMS)), ["%g" % l for l in LAMS])
axes[1].set_yticks(range(len(ALPHAS)), ["%g" % a for a in ALPHAS])
axes[1].set_xlabel("λ")
axes[1].set_ylabel("α")
axes[1].set_title("cv_score on the grid (lighter = better)")
ia, il = ALPHAS.index(ALPHA), LAMS.index(LAM)
axes[1].scatter([il], [ia], s=180, facecolors="none", edgecolors="red", linewidths=2)
fig.colorbar(im, ax=axes[1], label="cv_score")

fig.tight_layout()
plt.savefig(FIG_DIR / "fig_cv_delta_method3.png", dpi=150, bbox_inches="tight")
plt.show()

## Appendix B. Synthetic data (known true `x`)

`make_synthetic` (shared code) builds a smooth `x` on the group's own graph, adds Gaussian noise
(`σ = 0.1·std(y)`) and 3 outliers, so RMSE against the true `x` can be measured. This is a property of the
generator, not necessarily of the real data.

### B.1. One-seed check: does Huber "recognise" the planted outliers?

In [ ]:
x_true, y_syn = make_synthetic(G, y, seed=0)
x_syn_hat, info_syn = solve_huber(G, y_syn, return_info=True, **FINAL_PARAMS)

planted = np.argsort(-np.abs(y_syn - x_true))[:3]       # 3 nodes farthest from x = planted outliers
print("planted outliers (largest |y−x|):", sorted(planted.tolist()))
print("strongly down-weighted nodes (w < 0.5):", np.flatnonzero(info_syn["weights"] < 0.5).tolist())
print("\nplanted nodes in detail:")
for i in planted:
    print("  node %2d: |y−x| = %5.2f | y = %6.3f | x_hat = %6.3f | w = %.2f"
          % (i, abs(y_syn[i] - x_true[i]), y_syn[i], x_syn_hat[i], info_syn["weights"][i]))
print("\nRMSE of y (no processing)     : %.4f" % rmse(y_syn, x_true))
print("RMSE of x_hat (method 3)      : %.4f" % rmse(x_syn_hat, x_true))

### B.2. RMSE over seeds 0–19

References: `x_hat = y` (no processing), Tikhonov with the same `λ`, and method 6 (the group's comparison
baseline).

In [ ]:
# Synthetic RMSE for four options, same 20 seeds.
base = np.array([rmse(make_synthetic(G, y, sd)[1], make_synthetic(G, y, sd)[0]) for sd in range(20)])
r_huber = synth_rmse(FINAL_PARAMS)
r_l2 = synth_rmse({"delta": np.inf, "lam": LAM})

import sys
sys.path.append(str(ROOT / "src" / "method_6"))
from method6_outliers import solve_two_stage
r_m6 = np.array([rmse(solve_two_stage(G, make_synthetic(G, y, sd)[1]), make_synthetic(G, y, sd)[0])
                 for sd in range(20)])

print("%-38s %s" % ("option", "synthetic RMSE (20 seeds)"))
print("%-38s %.4f ± %.4f" % ("x_hat = y (no processing)", base.mean(), base.std()))
print("%-38s %.4f ± %.4f" % ("Huber (δ = %.3f, λ = %.1f)" % (FINAL_PARAMS["delta"], LAM),
                            r_huber.mean(), r_huber.std()))
print("%-38s %.4f ± %.4f" % ("Tikhonov with the same λ = %.1f" % LAM, r_l2.mean(), r_l2.std()))
print("%-38s %.4f ± %.4f" % ("Method 6 (two-stage)", r_m6.mean(), r_m6.std()))

diff = r_huber - r_l2
print("\nHuber − Tikhonov (paired per seed): %+.4f ± %.4f (standard error %.4f)"
      % (diff.mean(), diff.std(), diff.std() / np.sqrt(len(diff))))
print("=> on the group's generator, Huber is about %.3f RMSE worse; see the discussion in Section 5." % diff.mean())

# If delta is brought closer to the generator's noise scale (alpha = 0.3) and lam is reduced:
r_small = synth_rmse({"delta": 0.3 * S0, "lam": 0.05})
r_l2_small = synth_rmse({"delta": np.inf, "lam": 0.05})
print("\nWith delta better matched to the synthetic noise (alpha = 0.3) and lam = 0.05:")
print("  Huber = %.4f ± %.4f | Tikhonov = %.4f ± %.4f"
      % (r_small.mean(), r_small.std(), r_l2_small.mean(), r_l2_small.std()))
print("  => when delta matches the noise scale, Huber is slightly better than Tikhonov on this generator.")

## Appendix C. Comparison with Tikhonov on 100 public graphs

Question: is the Huber part **generally** useful, or only for the group's dataset? For each graph in
`public_cases.json`, choose `λ ∈ {0.05, 0.1, 0.2}` separately for each method by `cv_score` (3 splits for
speed), keep `α = 0.5` for Huber (with `s0` computed on that graph), then compare. This is a fairer comparison
than using one shared `λ`, because each graph has a different noise level.

In [ ]:
ALL = json.loads((ROOT / "public_cases.json").read_text(encoding="utf-8"))["cases"]

def load_case(cid):
    c = ALL[cid]
    Gc = nx.Graph()
    for i, xy in enumerate(c["positions"]):
        Gc.add_node(i, pos=tuple(map(float, xy)))
    for u, v, w in c["edges"]:
        Gc.add_edge(int(u), int(v), weight=float(w))
    return Gc, np.asarray(c["y"], dtype=float)

def cv_on(Gc, yc, params, seeds=range(3)):
    # cv_score for an arbitrary graph (same definition as the shared code).
    nn = len(yc)
    out = []
    for sd in seeds:
        folds = np.random.default_rng(sd).permutation(nn) % 5
        err = []
        for f in range(5):
            mk = folds != f
            err.append(np.abs(solve_huber(Gc, yc, mask=mk, **params)[~mk] - yc[~mk]))
        out.append(np.median(np.concatenate(err)))
    return float(np.mean(out))

LAMS_C = [0.05, 0.1, 0.2]
rows_c = []
for cid in sorted(ALL):
    Gc, yc = load_case(cid)
    s0c = noise_scale(Gc, yc)
    c_h = {l: cv_on(Gc, yc, {"delta": 0.5 * s0c, "lam": l}) for l in LAMS_C}
    c_l = {l: cv_on(Gc, yc, {"delta": np.inf, "lam": l}) for l in LAMS_C}
    bh, bl = min(c_h, key=c_h.get), min(c_l, key=c_l.get)
    rows_c.append({"cid": cid, "huber": c_h[bh], "l2": c_l[bl], "lam_h": bh, "lam_l": bl})

h = np.array([r["huber"] for r in rows_c])
l2 = np.array([r["l2"] for r in rows_c])
print("Number of graphs:", len(rows_c))
print("mean cv_score — Huber: %.4f | Tikhonov: %.4f | median: %.4f vs %.4f"
      % (h.mean(), l2.mean(), np.median(h), np.median(l2)))
print("Huber is better than Tikhonov on %d/%d graphs" % (int((h < l2).sum()), len(h)))
print("best λ chosen: Huber", {v: sum(1 for r in rows_c if r["lam_h"] == v) for v in LAMS_C},
      "| Tikhonov", {v: sum(1 for r in rows_c if r["lam_l"] == v) for v in LAMS_C})
print("median difference (Huber − Tikhonov): %+.4f" % np.median(h - l2))

## Appendix D. Convergence, correctness and noise level

Four checks: (1) IRLS gives the same solution as the convex L-BFGS solver on the exact objective; (2) `δ = ∞`
matches the closed-form Tikhonov solution, already checked in Section 2; (3) the notebook and the
`method3_huber.py` module give the same result; (4) a diagnostic of the real noise level from the spectral
structure of `y` (for explanation only, not used to choose parameters).

In [ ]:
import sys

from scipy.optimize import minimize

# (1) Compare with the convex L-BFGS solver on the same F(x).
f = lambda z: huber_objective(G, y, z, FINAL_PARAMS["delta"], LAM)
res = minimize(f, y, method="L-BFGS-B", options={"maxiter": 50000, "ftol": 1e-16, "gtol": 1e-12})
print("F(x_hat) = %.10f | L-BFGS = %.10f | difference = %.2e"
      % (f(x_hat_real), res.fun, abs(f(x_hat_real) - res.fun)))
print("max|IRLS − L-BFGS| = %.2e  (L-BFGS stops at gtol, hence the 1e-6 gap)"
      % np.max(np.abs(x_hat_real - res.x)))

# (2) The notebook and the shared module must give the same result.
sys.path.append(str(ROOT / "src" / "method_3"))
from method3_huber import solve_huber as solve_huber_module
assert np.allclose(solve_huber_module(G, y, **FINAL_PARAMS), x_hat_real, atol=1e-12)
print("notebook == module src/method_3/method3_huber.py: OK")

# (3) Noise diagnostic: the part of y outside the 5 lowest modes.
ev, U = np.linalg.eigh(L)
coef = U.T @ y
share = (coef[:5] ** 2).sum() / (coef ** 2).sum()
print("\nEnergy of y in the 5 lowest modes: %.2f%%" % (100 * share))
print("Standard deviation of the part outside the 5 modes (if the true x has only 5 modes): %.3f"
      % np.sqrt(np.mean(coef[5:] ** 2)))
print("σ of the synthetic generator = 0.1·std(y) = %.3f" % (0.1 * np.std(y)))
print("=> the real data is noisier than the group's generator, so synthetic RMSE does not directly predict real RMSE.")